# Kartverket municipality neighbors

The original Kartverket ZIP/GeoJSON is downloaded manually and kept unchanged in `data/bronze/kartverket/`. This notebook generates a derived municipality-neighbor mapping in Silver.

In [ ]:
from pathlib import Path
import json
from silver_layer.generate_neighbors import (
    generate_neighbors,
    PROJECT_ROOT,
    BRONZE_ROOT,
)

print("Project root:", PROJECT_ROOT)

Project root: /home/roland/UiS/rewrite/group14
Available Kartverket files:
 - Basisdata_0000_Norge_25833_Kommuner_GeoJSON.zip


## Select the downloaded file

Set `filename` to the actual ZIP or GeoJSON filename from the cell above.

In [4]:
filename = "Basisdata_0000_Norge_25833_Kommuner_GeoJSON.zip"
input_path = BRONZE_ROOT / filename
assert input_path.is_file(), f"File not found: {input_path}"

## Generate Silver mapping

In [5]:
output_path = generate_neighbors(input_path=input_path)
print("Output:", output_path)

Saved 357 municipalities to /home/roland/UiS/rewrite/group14/data/silver/geonorge/administrative_enheter_kommuner/municipality_neighbors.json
Output: /home/roland/UiS/rewrite/group14/data/silver/geonorge/administrative_enheter_kommuner/municipality_neighbors.json


## Inspect and validate results

In [6]:
with output_path.open(encoding="utf-8") as file:
    neighbors = json.load(file)

print("Municipalities:", len(neighbors))
print("Unique neighbor pairs:", sum(map(len, neighbors.values())) // 2)
print("Oslo (0301):", neighbors.get("0301"))
print(
    "Symmetric:",
    all(a in neighbors.get(b, []) for a, items in neighbors.items() for b in items),
)

Municipalities: 357
Unique neighbor pairs: 966
Oslo (0301): ['3201', '3205', '3207', '3212', '3220', '3222', '3232', '3234', '3305']
Symmetric: True


In [7]:
metadata_path = output_path.with_name(output_path.name + ".metadata.json")
with metadata_path.open(encoding="utf-8") as file:
    metadata = json.load(file)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

{
  "run_id": "1690381517a8407fa1e4f90b497dbaeb",
  "source": "Kartverket / Geonorge",
  "dataset": "geonorge/administrative_enheter_kommuner",
  "source_file": "/home/roland/UiS/rewrite/group14/data/bronze/kartverket/Basisdata_0000_Norge_25833_Kommuner_GeoJSON.zip",
  "source_sha256": "f4bb733477e033e16f98eb3382e35b2b95e81e0038baa8e28267277e62112735",
  "generated_at_utc": "2026-11-18T11:56:00.167662+00:00",
  "filename": "municipality_neighbors.json",
  "sha256": "3906c7af58d3c1dfeb335fc3107f2256e2909a05f383f18bbcb219a7b7564b50",
  "size_bytes": 28899,
  "coordinate_reference_system": "EPSG:25833",
  "adjacency_definition": "Geometries touch, including point-only contacts",
  "municipality_count": 357,
  "unique_adjacency_pair_count": 966,
  "municipality_codes_without_neighbors": [],
  "parameters": {
    "code_column": "kommunenummer",
    "type_column": "objtype",
    "municipality_type": "Kommune"
  }
}
